**Entregable 1** — Consolidación de Datos con Python

1.1 Leer el archivo y consolidar

In [ ]:
# Importación de liberias Python
from google.colab import drive
import pandas as pd

# Monta tu Drive (te pedirá autorización en una ventana)
drive.mount('/content/drive')

# Lee el archivo desde tu Drive (descárgalo primero como .xlsx)
ruta = "/content/drive/MyDrive/Service_centers.csv"
df = pd.read_csv(ruta)

# Imprimimos la info (Columnas, Tipo de dato) del Data Frame
# df.info()

# Casteamos la fecha de apertura
df["fecha_estimada_apertura"] = (
  pd.to_datetime(df["fecha_estimada_apertura"], errors="coerce")
  .dt.strftime("%Y-%m-%d")
)

# Calcula el promedio de las columnas 'obra_civil','compras','licencias' de forma horizontal
df["avance_total"] = df[["avance_obra_civil_pct","avance_compras_pct","avance_licencias_pct"]].mean(axis=1)

# Calcula los dias de retraso tomando la fecha actual
df["dias_retraso"] = (pd.to_datetime("today") - pd.to_datetime(df["fecha_estimada_apertura"])).dt.days

# Seteamos a 0 todos los valores negativos en dias_retraso
df["dias_retraso"] = df["dias_retraso"].clip(lower=0)

# Seleccionamos la columnas a trabajar y sumarizamos dias_retraso si aplica
df_resumen = df.groupby(["id_sc", "estado", "tipo_sc", "avance_obra_civil_pct",
                         "avance_compras_pct", "avance_licencias_pct", "fecha_estimada_apertura",
                         "avance_total"])["dias_retraso"].sum().reset_index()

# print(df_resumen)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


1.2 Exportamos los datos a un excel con Formato (no pude usar gspread por restriccion de APIS)

In [ ]:
# Importamos las librerias para formatear
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# Funcion para formatear el excel
def exportar_excel_formateado(df, ruta_drive):

    wb = Workbook()
    ws = wb.active
    ws.title = "Resumen"

    # ── Estilos ───────────────────────────────────────────────
    color_header     = "434343"
    color_fila_par   = "FEF5D9"
    color_fila_impar = "FEF0C7"

    estilo_borde = Border(
        left=Side(style="thin"),
        right=Side(style="thin"),
        top=Side(style="thin"),
        bottom=Side(style="thin")
    )

    # ── Encabezados ───────────────────────────────────────────
    for col_idx, col_nombre in enumerate(df.columns, start=1):
        celda = ws.cell(row=1, column=col_idx, value=col_nombre)
        celda.font      = Font(bold=True, color="FFFFFF", name="Arial", size=11)
        celda.fill      = PatternFill("solid", start_color=color_header)
        celda.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        celda.border    = estilo_borde

    ws.row_dimensions[1].height = 30

    # ── Datos ─────────────────────────────────────────────────
    for row_idx, row in enumerate(df.itertuples(index=False), start=2):
        color_fila = color_fila_par if row_idx % 2 == 0 else color_fila_impar
        for col_idx, valor in enumerate(row, start=1):
            celda = ws.cell(row=row_idx, column=col_idx, value=valor)
            celda.font      = Font(name="Arial", size=10)
            celda.fill      = PatternFill("solid", start_color=color_fila)
            celda.alignment = Alignment(horizontal="center", vertical="center")
            celda.border    = estilo_borde

            # Formato de 2 decimales para columnas numéricas
            if pd.api.types.is_numeric_dtype(df.iloc[:, col_idx - 1]):
                celda.number_format = "#,##0.00"

    # ── Ancho de columnas automático ──────────────────────────
    for col_idx, col_nombre in enumerate(df.columns, start=1):
        max_ancho = max(
            len(str(col_nombre)),
            df.iloc[:, col_idx - 1].astype(str).map(len).max()
        )
        ws.column_dimensions[get_column_letter(col_idx)].width = min(max_ancho + 4, 40)

    wb.save(ruta_drive)
    print(f"✅ Archivo guardado en: {ruta_drive}")


# Guadado del archivo en Drive
ruta = "/content/drive/MyDrive/Service_Centers_ETL.xlsx"
exportar_excel_formateado(df_resumen, ruta)

✅ Archivo guardado en: /content/drive/MyDrive/Service_Centers_ETL.xlsx


**Entregable 3** — Módulo de IA para Detección de Riesgos

In [ ]:
import pandas as pd
from openai import OpenAI
from google.colab import userdata

client = OpenAI(api_key=userdata.get('Api_GPT'))

# Convierte el DataFrame a JSON
df_json = df_resumen.to_json(orient="records", force_ascii=False, indent=2)

# Arma el mensaje incluyendo el JSON
prompt = "Eres analista de operaciones logísticas. Analiza el siguiente reporte de aperturas de Service Centers y genera: 1. Un párrafo de resumen ejecutivo (máximo 80 palabras). 2. Los 3 principales riesgos operativos esta semana. 3. Una acción recomendada por cada riesgo."

respuesta = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {
            "role": "system",
            "content": "Eres un analista de datos. Recibirás datos en formato JSON y responderás preguntas sobre ellos."
        },
        {
            "role": "user",
            "content": f"{prompt}\n\nDatos:\n{df_json}"
        }
    ]
)

print(respuesta.choices[0].message.content)


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

**4.- Github**

In [50]:
from google.colab import userdata
import os
import requests

# ── Variables ─────────────────────────────────────────────────
GITHUB_TOKEN = userdata.get("Github")   # Tu token de GitHub en Secrets
GITHUB_USER  = userdata.get("User")    # Tu usuario de GitHub en Secrets
REPO_NOMBRE  = "Worksample_SetUp"

# ── 2. Copiar el notebook y subir ─────────────────────────────
!cp "/content/drive/MyDrive/Worksample_SetUp.ipynb" /content/

os.chdir("/content")
!git config --global user.email "alonso.hernandez@mercadolibre.com.mx"
!git config --global user.name "alonsohernandez-spec"
!git init
!git add "Worksample_SetUp.ipynb"
!git commit -m "feat: agrega ETL Service Centers y módulo IA"
!git branch -M main

# Línea corregida — las variables se pasan como f-string en Python, no dentro del !
!git remote remove origin
!git remote add origin https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NOMBRE}.git


print("✅ Repositorio creado y código subido")

Reinitialized existing Git repository in /content/.git/
Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@6c0710dc25e6.(none)')
error: remote origin already exists.
error: src refspec main does not match any
error: failed to push some refs to 'https://github.com/alonsohernandez-spec/Worksample_SetUp.git'
✅ Repositorio creado y código subido
